# NeuroGolf 2026: First ONNX Submission

This tutorial is for a Kaggle participant who wants a fast, correct first foothold in the 2026 NeuroGolf Championship.

By the end you will be able to:

- locate the official NeuroGolf task JSON files in a Kaggle notebook,
- understand the unusual submission format (`submission.zip`, not `submission.csv`),
- build tiny ONNX networks for simple ARC color-map transformations,
- validate those color-map rules against the provided train, test, and `arc-gen` examples,
- create a first `submission.zip` that Kaggle can score.

Outline:

1. Inspect the competition data and task schema.
2. Represent ARC grids as NeuroGolf one-hot tensors.
3. Build a minimal 1x1 convolution ONNX network.
4. Validate several color-map tasks.
5. Package the ONNX files into `submission.zip`.
6. Extend the idea to richer tasks.
    

## 1. Setup

The official competition data is attached as a competition source. On Kaggle it is usually available at `/kaggle/input/neurogolf-2026`. Locally, this notebook also works from the repository checkout if `competitions/neurogolf-2026/data/` has been downloaded.
    

In [1]:
from __future__ import annotations

import json
import math
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, helper

DATA_CANDIDATES = [
    Path('/kaggle/input/neurogolf-2026'),
    Path('/kaggle/input/competitions/neurogolf-2026'),
    Path('../data'),
    Path('competitions/neurogolf-2026/data'),
    Path('data'),
]

DATA_DIR = next((path for path in DATA_CANDIDATES if (path / 'task001.json').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not find NeuroGolf task JSONs. Attach the competition data or download it locally.')

IS_KAGGLE = Path('/kaggle/working').exists()
WORK_DIR = Path('/kaggle/working/neurogolf_tutorial') if IS_KAGGLE else Path('../submissions/tutorial-color-map-baseline')
WORK_DIR.mkdir(parents=True, exist_ok=True)
NETWORK_DIR = WORK_DIR / 'networks'
ZIP_PATH = WORK_DIR / 'submission.zip'

print(f'Data directory: {DATA_DIR}')
print(f'Working directory: {WORK_DIR}')
print(f'ONNX: {onnx.__version__}')
    

Data directory: ../data
Working directory: ../submissions/tutorial-color-map-baseline
ONNX: 1.22.0


## 2. Inspect one task

Each task has `train`, `test`, and `arc-gen` examples. The public `test` examples are not a leaderboard test set in the usual Kaggle sense; they are part of the task definition and can be used for validation. Kaggle also checks private examples, so the goal is to express the rule, not memorize outputs.
    

In [2]:
def load_task(task_num: int) -> dict:
    with (DATA_DIR / f'task{task_num:03d}.json').open() as f:
        return json.load(f)

sample = load_task(16)
print(sample.keys())
for split, examples in sample.items():
    first = examples[0]
    input_shape = (len(first['input']), len(first['input'][0]))
    output_shape = (len(first['output']), len(first['output'][0]))
    print(f'{split:7s}: {len(examples):3d} examples, first shape {input_shape} -> {output_shape}')

print('First task016 input:')
print(np.array(sample['train'][0]['input']))
print('First task016 output:')
print(np.array(sample['train'][0]['output']))
    

dict_keys(['train', 'test', 'arc-gen'])
train  :   4 examples, first shape (3, 3) -> (3, 3)
test   :   1 examples, first shape (3, 3) -> (3, 3)
arc-gen: 262 examples, first shape (3, 3) -> (3, 3)
First task016 input:
[[3 1 2]
 [3 1 2]
 [3 1 2]]
First task016 output:
[[4 5 6]
 [4 5 6]
 [4 5 6]]


## 3. Convert grids to NeuroGolf tensors

NeuroGolf models take one tensor named `input` and return one tensor named `output`. The shape is always `[1, 10, 30, 30]`: one batch, ten color channels, and a padded 30x30 grid. A cell is represented as one-hot color channels.
    

In [3]:
BATCH, CHANNELS, HEIGHT, WIDTH = 1, 10, 30, 30
GRID_SHAPE = [BATCH, CHANNELS, HEIGHT, WIDTH]


def grid_to_onehot(grid: list[list[int]]) -> np.ndarray:
    arr = np.zeros(GRID_SHAPE, dtype=np.float32)
    if len(grid) > HEIGHT or len(grid[0]) > WIDTH:
        raise ValueError('This helper only handles grids up to 30x30, matching the competition helper.')
    for row_idx, row in enumerate(grid):
        for col_idx, color in enumerate(row):
            arr[0, int(color), row_idx, col_idx] = 1.0
    return arr


def onehot_to_grid(arr: np.ndarray, rows: int, cols: int) -> list[list[int]]:
    logits = arr[0, :, :rows, :cols]
    return logits.argmax(axis=0).astype(int).tolist()

example = sample['train'][0]
tensor = grid_to_onehot(example['input'])
print(tensor.shape, tensor.dtype, int(tensor.sum()))
print(onehot_to_grid(tensor, len(example['input']), len(example['input'][0])) == example['input'])
    

(1, 10, 30, 30) float32 9
True


## 4. Build a tiny color-map network

Some ARC tasks keep the same grid shape and only recolor cells. A 1x1 convolution can express this exactly: each input color channel maps to one output color channel at the same row and column.

This is a deliberately small starting point. It will not solve most tasks, but it teaches the NeuroGolf submission mechanics and creates a valid package.
    

In [4]:
IR_VERSION = 10
OPSET_IMPORTS = [helper.make_opsetid('', 10)]


def make_color_map_network(color_map: dict[int, int]) -> onnx.ModelProto:
    weights = np.zeros((CHANNELS, CHANNELS, 1, 1), dtype=np.float32)
    for input_color in range(CHANNELS):
        output_color = color_map.get(input_color, input_color)
        weights[output_color, input_color, 0, 0] = 1.0

    x = helper.make_tensor_value_info('input', TensorProto.FLOAT, GRID_SHAPE)
    y = helper.make_tensor_value_info('output', TensorProto.FLOAT, GRID_SHAPE)
    w = helper.make_tensor('W', TensorProto.FLOAT, weights.shape, weights.flatten().tolist())
    node = helper.make_node('Conv', ['input', 'W'], ['output'], kernel_shape=[1, 1], pads=[0, 0, 0, 0])
    graph = helper.make_graph([node], 'color_map_graph', [x], [y], [w])
    model = helper.make_model(graph, ir_version=IR_VERSION, opset_imports=OPSET_IMPORTS)
    onnx.checker.check_model(model)
    return model


def apply_color_map_to_grid(grid: list[list[int]], color_map: dict[int, int]) -> list[list[int]]:
    return [[color_map.get(int(color), int(color)) for color in row] for row in grid]


def count_params(model: onnx.ModelProto) -> int:
    return sum(math.prod(init.dims) for init in model.graph.initializer)

identity_model = make_color_map_network({color: color for color in range(CHANNELS)})
print(f'Identity model initializers: {len(identity_model.graph.initializer)}')
print(f'Identity model params: {count_params(identity_model)}')


Identity model initializers: 1
Identity model params: 100


## 5. Validate candidate tasks

The following four tasks were found by scanning all 400 task files for transformations that are pure per-cell color maps across `train`, `test`, and `arc-gen`. Before packaging, we still validate every example.
    

In [5]:
TASK_COLOR_MAPS = {
    16: {0: 0, 1: 5, 2: 6, 3: 4, 4: 3, 5: 1, 6: 2, 7: 7, 8: 9, 9: 8},
    276: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 2, 7: 7, 8: 8, 9: 9},
    309: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 5, 8: 8, 9: 9},
    337: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 8, 6: 6, 7: 7, 8: 5, 9: 9},
}


def validate_task(task_num: int, color_map: dict[int, int]) -> tuple[int, int]:
    task = load_task(task_num)
    passed = 0
    failed = 0
    for split in ['train', 'test', 'arc-gen']:
        for example in task[split]:
            pred_grid = apply_color_map_to_grid(example['input'], color_map)
            if pred_grid == example['output']:
                passed += 1
            else:
                failed += 1
    return passed, failed

results = []
for task_num, mapping in TASK_COLOR_MAPS.items():
    model = make_color_map_network(mapping)
    passed, failed = validate_task(task_num, mapping)
    results.append({'task': f'task{task_num:03d}', 'passed_examples': passed, 'failed_examples': failed, 'params': count_params(model)})

for row in results:
    print(row)

assert all(row['failed_examples'] == 0 for row in results), 'Do not package a task that fails local validation.'


{'task': 'task016', 'passed_examples': 267, 'failed_examples': 0, 'params': 100}
{'task': 'task276', 'passed_examples': 266, 'failed_examples': 0, 'params': 100}
{'task': 'task309', 'passed_examples': 265, 'failed_examples': 0, 'params': 100}
{'task': 'task337', 'passed_examples': 266, 'failed_examples': 0, 'params': 100}


## 6. Package `submission.zip`

The competition expects a zip file containing root-level ONNX files named `task001.onnx` through `task400.onnx`. You can include fewer than 400 files; omitted tasks simply earn no points.

This cell writes a small, valid package for the four validated color-map tasks.
    

In [6]:
if NETWORK_DIR.exists():
    shutil.rmtree(NETWORK_DIR)
NETWORK_DIR.mkdir(parents=True, exist_ok=True)

for task_num, mapping in TASK_COLOR_MAPS.items():
    model = make_color_map_network(mapping)
    passed, failed = validate_task(task_num, mapping)
    if failed:
        raise RuntimeError(f'task{task_num:03d} failed validation: {failed} failures')
    onnx.save(model, NETWORK_DIR / f'task{task_num:03d}.onnx')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for onnx_path in sorted(NETWORK_DIR.glob('task*.onnx')):
        zf.write(onnx_path, arcname=onnx_path.name)

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()

print(f'Created: {ZIP_PATH}')
print(f'Zip entries: {names}')
print(f'Zip size: {ZIP_PATH.stat().st_size} bytes')
assert all('/' not in name for name in names), 'ONNX files must be at the zip root.'
    

Created: ../submissions/tutorial-color-map-baseline/submission.zip
Zip entries: ['task016.onnx', 'task276.onnx', 'task309.onnx', 'task337.onnx']
Zip size: 1058 bytes


## 7. What this baseline teaches

This package is intentionally modest. Its value is that it exercises the whole competition loop: task inspection, ONNX graph construction, validation, and zip packaging.

Common pitfall: do not create a `submission.csv`; Kaggle will not score it for this competition. The scored artifact is `submission.zip` with ONNX files at the archive root.

Next extensions:

- scan same-shape tasks for simple local-neighborhood rules expressible as 3x3 convolutions,
- add fixed flips, shifts, crops, and masks as separate ONNX graph templates,
- build a task classifier that groups the 400 ARC tasks by shape relation and color behavior,
- submit this package first as an anchor, then improve coverage with new task templates.
    

## Exercise

Pick one same-shape task that is not in `TASK_COLOR_MAPS`. Check whether it is a pure color map by building a mapping from input cell color to output cell color over all examples.

Answer scaffold: fill in `task_num` and run the next cell. If `consistent` is `True`, add the mapping to `TASK_COLOR_MAPS` and rerun validation.
    

In [7]:
# Exercise scaffold
task_num = 10  # Try another task number here.
task = load_task(task_num)
mapping = {}
consistent = True

for split in ['train', 'test', 'arc-gen']:
    for example in task[split]:
        input_grid = example['input']
        output_grid = example['output']
        if (len(input_grid), len(input_grid[0])) != (len(output_grid), len(output_grid[0])):
            consistent = False
            break
        for r, row in enumerate(input_grid):
            for c, input_color in enumerate(row):
                output_color = output_grid[r][c]
                previous = mapping.get(input_color)
                if previous is not None and previous != output_color:
                    consistent = False
                mapping[input_color] = output_color
        if not consistent:
            break
    if not consistent:
        break

print(f'task{task_num:03d} pure color map? {consistent}')
print(mapping)
    

task010 pure color map? False
{0: 0, 5: 4}


## Manual submission

After reviewing the generated zip in this notebook, submit it from the Kaggle competition page or with the Kaggle CLI:

```bash
kaggle competitions submit -c neurogolf-2026 -f /kaggle/working/neurogolf_tutorial/submission.zip -m "tutorial color-map baseline"
```

For this review notebook, I recommend inspecting the output first rather than submitting automatically.
    